# 01 · Inspección de la base de comercio de calzado (España, 2020-2024)

**Objetivo.** Verificar la calidad, estructura e integridad de la base antes de cualquier análisis: tipos de datos, valores faltantes, duplicados, cobertura temporal, presencia de registros agregados ("Mundo") y coherencia entre las columnas numéricas.

**Recordatorio metodológico.** La base es de **España** y se usa como **benchmark externo** para Manaco. No contiene datos internos de la empresa.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

sys.path.append(str(Path.cwd().parent / "src"))
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 200
pd.options.display.max_columns = 30

from carga import cargar, resumen_calidad, CLAVE

df, mundo = cargar()
resumen_calidad(df)
df.head()

RESUMEN DE CALIDAD DE DATOS
Filas: 240
Columnas: 10

Nulos por columna:
País_reportante      0
Flujo_comercial      0
Año                  0
Código_SA6           0
Categoría_calzado    0
Socio_comercial      0
Valor_miles_USD      0
Cantidad_pares       0
Valor_USD            0
Valor_USD_por_par    0

Años cubiertos: [2020, 2021, 2022, 2023, 2024]
Países socios únicos: 30
Categorías SA6 únicas: ['640399', '640411', '640420']
Filas con clave repetida (flujo, año, SA6, socio): 20


,País_reportante,Flujo_comercial,Año,Código_SA6,Categoría_calzado,Socio_comercial,Valor_miles_USD,Cantidad_pares,Valor_USD,Valor_USD_por_par
0,España,Importación,2023,640411,Calzado deportivo con suela de caucho/plástico,Indonesia,"62,576.67","2,075,438.00","62,576,674.41",30.15
1,España,Importación,2024,640411,Calzado deportivo con suela de caucho/plástico,Estados Unidos,"36,379.51","1,199,546.00","36,379,513.62",30.33
2,España,Importación,2022,640411,Calzado deportivo con suela de caucho/plástico,Grecia,"853,548.06","30,537,251.00","853,548,057.31",27.95
3,España,Importación,2020,640411,Calzado deportivo con suela de caucho/plástico,Eslovaquia,"166,572.63","4,861,274.00","166,572,626.51",34.27
4,España,Importación,2023,640411,Calzado deportivo con suela de caucho/plástico,Italia,"241,958.40","11,767,364.00","241,958,400.09",20.56


## Diccionario de datos

| Columna | Tipo (tras la carga) | Unidad | Descripción |
|---|---|---|---|
| País_reportante | texto | — | País que reporta el comercio (España) |
| Flujo_comercial | texto | — | Importación o Exportación |
| Año | entero | año | Periodo 2020-2024 |
| Código_SA6 | texto | — | Código del Sistema Armonizado a 6 dígitos (640411, 640399, 640420) |
| Categoría_calzado | texto | — | Descripción de la categoría SA6 según la base |
| Socio_comercial | texto | — | País de origen (importación) o de destino (exportación) |
| Valor_miles_USD | decimal | miles de USD | Valor comercial |
| Cantidad_pares | decimal | pares | Volumen de pares |
| Valor_USD | decimal | USD | Valor_miles_USD × 1000 |
| Valor_USD_por_par | decimal | USD por par | Valor_USD / Cantidad_pares (valor aduanero, **no** precio al consumidor) |

In [2]:
print("Valores faltantes por columna:")
print(df.isna().sum().to_string())

print("\nFilas duplicadas exactas:", int(df.duplicated().sum()))

repetidas = df[df.duplicated(CLAVE, keep=False)].sort_values(CLAVE)
print(f"Filas con clave repetida {CLAVE}: {len(repetidas)} ({len(repetidas) // 2} pares aprox.)")
display(repetidas[CLAVE + ["Valor_USD", "Cantidad_pares", "Valor_USD_por_par"]].head(8))

print("\nAños cubiertos:", sorted(df["Año"].unique().tolist()))
print("Rango de años:", df["Año"].min(), "a", df["Año"].max())

print("\nRegistros 'Mundo' en la base:", int((df["Socio_comercial"].str.lower() == "mundo").sum()),
      "| separados por cargar():", len(mundo))

print("\nRegistros por categoría y flujo:")
print(df.groupby(["Código_SA6", "Flujo_comercial"]).size().unstack().to_string())

print("\nRegistros por flujo y año:")
print(df.groupby(["Flujo_comercial", "Año"]).size().unstack().to_string())

Valores faltantes por columna:
País_reportante      0
Flujo_comercial      0
Año                  0
Código_SA6           0
Categoría_calzado    0
Socio_comercial      0
Valor_miles_USD      0
Cantidad_pares       0
Valor_USD            0
Valor_USD_por_par    0

Filas duplicadas exactas: 0
Filas con clave repetida ['Flujo_comercial', 'Año', 'Código_SA6', 'Socio_comercial']: 20 (10 pares aprox.)


,Flujo_comercial,Año,Código_SA6,Socio_comercial,Valor_USD,Cantidad_pares,Valor_USD_por_par
48,Exportación,2020,640411,Estados Unidos,"21,696,383.11","901,775.00",24.06
78,Exportación,2020,640411,Estados Unidos,"25,633,204.33","1,584,001.00",16.18
120,Exportación,2023,640399,Austria,"255,074,307.70","12,134,129.00",21.02
150,Exportación,2023,640399,Austria,"69,998,948.56","1,255,657.00",55.75
202,Exportación,2023,640420,Bélgica,"2,620,350.49","50,578.00",51.81
232,Exportación,2023,640420,Bélgica,"2,270,076.17","76,929.00",29.51
124,Exportación,2024,640399,Grecia,"207,254,466.07","9,032,216.00",22.95
154,Exportación,2024,640399,Grecia,"213,992,951.96","5,427,396.00",39.43



Años cubiertos: [2020, 2021, 2022, 2023, 2024]
Rango de años: 2020 a 2024

Registros 'Mundo' en la base: 0 | separados por cargar(): 0

Registros por categoría y flujo:
Flujo_comercial  Exportación  Importación
Código_SA6                               
640399                    40           40
640411                    40           40
640420                    40           40

Registros por flujo y año:
Año              2020  2021  2022  2023  2024
Flujo_comercial                              
Exportación        15    19    18    26    42
Importación        13    15    28    24    40


In [3]:
dif_valor_rel = ((df["Valor_miles_USD"] * 1000 - df["Valor_USD"]).abs() / df["Valor_USD"])
valor_par_calc = df["Valor_USD"] / df["Cantidad_pares"]
dif_par = (valor_par_calc - df["Valor_USD_por_par"]).abs()

print(f"Máxima diferencia relativa Valor_miles_USD*1000 vs Valor_USD: {dif_valor_rel.max():.2e}")
print(f"Máxima diferencia absoluta Valor_USD/Cantidad_pares vs Valor_USD_por_par: {dif_par.max():.4f} USD")
print("¿Todo coherente (tolerancias de redondeo)?", bool((dif_valor_rel < 1e-3).all() and (dif_par < 0.01).all()))
print("¿Valores y cantidades estrictamente positivos?",
      bool((df[["Valor_USD", "Cantidad_pares", "Valor_USD_por_par"]] > 0).all().all()))

print("\nRangos de las variables numéricas:")
print(df[["Valor_miles_USD", "Cantidad_pares", "Valor_USD_por_par"]].describe().round(2).to_string())

Máxima diferencia relativa Valor_miles_USD*1000 vs Valor_USD: 3.83e-05
Máxima diferencia absoluta Valor_USD/Cantidad_pares vs Valor_USD_por_par: 0.0051 USD
¿Todo coherente (tolerancias de redondeo)? True
¿Valores y cantidades estrictamente positivos? True

Rangos de las variables numéricas:
       Valor_miles_USD  Cantidad_pares  Valor_USD_por_par
count           240.00          240.00             240.00
mean        122,597.67    4,841,085.69              36.41
std         208,513.36    9,411,244.86              17.33
min              95.61        4,951.00               8.00
25%           6,963.30      181,425.25              24.36
50%          36,869.04    1,235,575.00              31.68
75%         156,617.34    5,083,497.50              43.76
max       1,688,184.60   79,245,283.00             119.24


## Conclusiones de la inspección

- **Calidad formal alta:** la base tiene 240 registros y 10 columnas, sin valores faltantes ni duplicados exactos; cubre 2020-2024 y 30 socios comerciales.
- **Sin doble conteo:** no existen registros "Mundo"; la separación prevista en `cargar()` queda como salvaguarda y no se activó.
- **Integridad numérica consistente:** `Valor_miles_USD × 1000` coincide con `Valor_USD` y `Valor_USD / Cantidad_pares` coincide con `Valor_USD_por_par`; las diferencias son de redondeo.
- **Atención a la estructura:** la base es una **muestra** (40 registros por combinación categoría × flujo) y hay 20 filas con clave repetida (flujo, año, SA6, socio) con valores distintos; se **conservan** porque no son duplicados exactos y podrían corresponder a registros distintos de la fuente. El número de registros por año varía, por lo que las comparaciones anuales deben leerse con cautela.